# GALVAT Trial 8: GALVAT-Optimized Training, Evaluation, and Export

This notebook restores the validated 44-class Trial 8 checkpoint, audits and locally
removes exact-image leakage, preprocesses every image to an auto-oriented 640x640
letterboxed training copy, trains YOLO26s on an A100, and selects the **checkpoint,
NMS setting, and per-class confidence thresholds using validation GALVAT weighted
score**. Test is evaluated once with frozen validation-selected settings.

Primary output: full-ontology GALVAT weighted score and a 44-row per-class report.
Standard AP, background-RoI false positives, spoken-output burden, PyTorch weights,
ONNX, thresholds, charts, and audit manifests are also saved to Drive.

The source ZIP is never modified. Exact duplicates are removed only from the restored
`/content` copy, with test taking precedence over validation and validation over train.

**Overnight use:** choose an A100 runtime and run Cell 1 once so Drive authorization
completes, then use **Runtime -> Run all**. The Drive run folder has a fixed name.
Training resumes from `last.pt`, completed validation pairs are cached, and the frozen
test result is locked. If Colab disconnects, reconnect and run all cells again.

In [ ]:
# Cell 1: Fixed environment, A100 guard, and run configuration
!pip -q install ultralytics==8.4.102 onnx onnxruntime pyyaml pandas seaborn tqdm pillow

from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Mapping, Sequence
import datetime, gc, hashlib, json, math, os, random, shutil, zipfile

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

import torch
import ultralytics
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
GPU_NAME = torch.cuda.get_device_name(0)
print('Ultralytics:', ultralytics.__version__)
print('Torch:', torch.__version__)
print('GPU:', GPU_NAME)
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
assert 'A100' in GPU_NAME.upper(), f'This run is configured for A100, found: {GPU_NAME}'

DATASET_ZIP = Path(
    '/content/drive/MyDrive/galvat_trial8_dataset_checkpoints/'
    'galvat_trial8_worker_manual_merged_2026-08-03_01-21-07.zip'
)
RESTORE_ROOT = Path('/content/galvat_trial8_training_restore')
DRIVE_RUNS = Path('/content/drive/MyDrive/galvat_trial8_training_runs')

RUN_STAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
# Stable name is intentional: rerunning all cells resumes the same Drive checkpoints.
RUN_NAME = 'galvat_trial8_yolo26s_640_galvat_selected_2026-08-03_v1'
RUN_ROOT = DRIVE_RUNS / RUN_NAME
AUDIT_DIR = RUN_ROOT / 'dataset_audit'
SELECTION_DIR = RUN_ROOT / 'validation_selection'
TEST_DIR = RUN_ROOT / 'test_frozen'
CHART_DIR = RUN_ROOT / 'charts'
EXPORT_DIR = RUN_ROOT / 'export'
for path in [RUN_ROOT, AUDIT_DIR, SELECTION_DIR, TEST_DIR, CHART_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_CLASSES = 44
EXPECTED_SOURCE_IMAGES = 34268
EXPECTED_SOURCE_BOXES = 59144
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

MODEL_INIT = 'yolo26s.pt'
IMGSZ = 640
EPOCHS = 120
SAVE_PERIOD = 10
MATCH_IOU = 0.50
NMS_IOU_VALUES = [0.55, 0.70, 0.80]
CONF_VALUES = [round(x, 2) for x in np.arange(0.05, 0.91, 0.05)]
PRED_MIN_CONF = 0.025
PRED_BATCH = 8
MAX_DET = 300

print('Dataset:', DATASET_ZIP)
print('Run root:', RUN_ROOT)

In [ ]:
# Cell 2: Restore only the shallow validated dataset from the ZIP
assert DATASET_ZIP.exists(), f'Missing checkpoint: {DATASET_ZIP}'

if RESTORE_ROOT.exists():
    shutil.rmtree(RESTORE_ROOT)
RESTORE_ROOT.mkdir(parents=True)

with zipfile.ZipFile(DATASET_ZIP, 'r') as archive:
    members = archive.namelist()
    yaml_members = [
        name for name in members
        if name.endswith('/dataset/data.yaml') or name == 'dataset/data.yaml'
    ]
    assert yaml_members, 'No dataset/data.yaml found in checkpoint.'

    def member_rank(name):
        parts = PurePosixPath(name).parts
        return (parts.count('restore'), len(parts), name)

    selected_yaml = sorted(yaml_members, key=member_rank)[0]
    dataset_member_root = PurePosixPath(selected_yaml).parent
    internal_root = dataset_member_root.parent
    prefix = dataset_member_root.as_posix().rstrip('/') + '/'
    selected_members = [name for name in members if name.startswith(prefix)]

    for name in tqdm(selected_members, desc='Extracting validated dataset'):
        member_path = PurePosixPath(name)
        assert '..' not in member_path.parts and not member_path.is_absolute()
        archive.extract(name, RESTORE_ROOT)

    summary_member = (internal_root / 'metadata' / 'checkpoint_summary.json').as_posix()
    checkpoint_summary = (
        json.loads(archive.read(summary_member))
        if summary_member in members else {}
    )

DATA_YAML = RESTORE_ROOT / Path(selected_yaml)
DATA_ROOT = DATA_YAML.parent
assert DATA_YAML.exists()

with DATA_YAML.open() as handle:
    data_cfg = yaml.safe_load(handle)
names_obj = data_cfg['names']
if isinstance(names_obj, dict):
    NAMES = [names_obj[i] if i in names_obj else names_obj[str(i)] for i in range(len(names_obj))]
else:
    NAMES = list(names_obj)

data_cfg.update({
    'path': str(DATA_ROOT),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {i: name for i, name in enumerate(NAMES)},
    'nc': len(NAMES),
})
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False))
name_to_id = {name: i for i, name in enumerate(NAMES)}

assert len(NAMES) == EXPECTED_CLASSES, NAMES
if checkpoint_summary:
    assert checkpoint_summary['final_images'] == EXPECTED_SOURCE_IMAGES
    assert checkpoint_summary['final_boxes'] == EXPECTED_SOURCE_BOXES
    assert checkpoint_summary['bad_final_label_rows'] == 0

print('Selected archive dataset:', selected_yaml)
print('DATA_ROOT:', DATA_ROOT)
print('Classes:', len(NAMES))
print(json.dumps(checkpoint_summary, indent=2))

In [ ]:
# Cell 3: Audit labels and remove exact-image leakage from the local copy only
SPLITS = ['train', 'valid', 'test']

def label_path_for_image(image_path):
    return image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'

def split_images(split):
    image_dir = DATA_ROOT / split / 'images'
    return sorted(p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

def parse_label_file(label_path, split='', image_name=''):
    rows, issues = [], []
    lines = label_path.read_text().splitlines() if label_path.exists() else []
    for line_number, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        parts = line.split()
        reason = None
        if len(parts) != 5:
            reason = 'bad_column_count'
        else:
            try:
                class_id = int(float(parts[0]))
                x, y, w, h = map(float, parts[1:])
            except Exception:
                reason = 'parse_error'
            else:
                if class_id < 0 or class_id >= len(NAMES):
                    reason = 'bad_class_id'
                elif not all(math.isfinite(v) for v in [x, y, w, h]):
                    reason = 'nonfinite_coordinate'
                elif not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    reason = 'bad_coordinates'
        if reason:
            issues.append({
                'split': split, 'file': image_name, 'line': line_number,
                'reason': reason, 'raw': line,
            })
        else:
            rows.append((class_id, x, y, w, h))
    return rows, issues

def sha1_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha1()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def xywh_to_xyxy(row):
    _, x, y, w, h = row
    return np.array([x - w / 2, y - h / 2, x + w / 2, y + h / 2], dtype=float)

initial_images = sum(len(split_images(split)) for split in SPLITS)
initial_boxes = 0
label_issues = []
hash_rows = []

for split in SPLITS:
    for image_path in tqdm(split_images(split), desc=f'Auditing {split}'):
        label_rows, issues = parse_label_file(label_path_for_image(image_path), split, image_path.name)
        initial_boxes += len(label_rows)
        label_issues.extend(issues)
        hash_rows.append({
            'split': split,
            'image': str(image_path),
            'label': str(label_path_for_image(image_path)),
            'sha1': sha1_file(image_path),
            'box_count': len(label_rows),
        })

assert initial_images == EXPECTED_SOURCE_IMAGES, (initial_images, EXPECTED_SOURCE_IMAGES)
assert initial_boxes == EXPECTED_SOURCE_BOXES, (initial_boxes, EXPECTED_SOURCE_BOXES)
assert not label_issues, pd.DataFrame(label_issues).head(50)

hash_df = pd.DataFrame(hash_rows)
duplicate_groups = hash_df.groupby('sha1').filter(lambda frame: len(frame) > 1)
cross_split_hashes = int((duplicate_groups.groupby('sha1')['split'].nunique() > 1).sum()) if len(duplicate_groups) else 0
print('Exact duplicate files:', len(duplicate_groups))
print('Exact duplicate hash groups crossing splits:', cross_split_hashes)

split_priority = {'train': 1, 'valid': 2, 'test': 3}
dedupe_rows = []
for sha1, group in tqdm(list(hash_df.groupby('sha1')), desc='Deduplicating exact images'):
    if len(group) == 1:
        continue
    records = group.to_dict('records')
    records.sort(key=lambda row: (-split_priority[row['split']], row['image']))
    keep = records[0]
    kept_rows, issues = parse_label_file(
        Path(keep['label']), keep['split'], Path(keep['image']).name
    )
    assert not issues
    kept_semantic_labels = sorted(kept_rows)
    for removed in records[1:]:
        removed_rows, issues = parse_label_file(
            Path(removed['label']), removed['split'], Path(removed['image']).name
        )
        assert not issues
        label_conflict = sorted(removed_rows) != kept_semantic_labels
        Path(removed['image']).unlink()
        Path(removed['label']).unlink(missing_ok=True)
        dedupe_rows.append({
            'sha1': sha1,
            'kept_split': keep['split'],
            'kept_image': keep['image'],
            'removed_split': removed['split'],
            'removed_image': removed['image'],
            'kept_box_count': len(kept_rows),
            'removed_box_count': len(removed_rows),
            'label_conflict': label_conflict,
            'resolution': 'kept_winner_label_unchanged',
        })

dedupe_df = pd.DataFrame(dedupe_rows)
dedupe_df.to_csv(AUDIT_DIR / 'exact_hash_dedupe_manifest.csv', index=False)
hash_df.to_csv(AUDIT_DIR / 'source_image_hash_manifest.csv', index=False)

# Establish an independent post-dedupe source-of-truth before preprocessing.
source_sha1_by_path = dict(zip(hash_df['image'], hash_df['sha1']))
post_dedupe_state = hashlib.sha1()
post_dedupe_state.update(json.dumps(NAMES, separators=(',', ':')).encode('utf-8'))
expected_image_counts = Counter()
expected_box_counts = Counter()
expected_split_images = {}
post_dedupe_rows = []
for split in SPLITS:
    images = split_images(split)
    expected_split_images[split] = len(images)
    for image_path in images:
        rows, issues = parse_label_file(label_path_for_image(image_path), split, image_path.name)
        assert not issues
        label_sha1 = sha1_file(label_path_for_image(image_path))
        image_sha1 = source_sha1_by_path[str(image_path)]
        post_dedupe_state.update(
            f'{split}|{image_path.name}|{image_sha1}|{label_sha1}\n'.encode('utf-8')
        )
        seen_classes = set()
        for class_id, *_ in rows:
            class_name = NAMES[class_id]
            expected_box_counts[(split, class_name)] += 1
            seen_classes.add(class_name)
        for class_name in seen_classes:
            expected_image_counts[(split, class_name)] += 1
        post_dedupe_rows.append({
            'split': split,
            'image': str(image_path),
            'image_sha1': image_sha1,
            'label_sha1': label_sha1,
            'box_count': len(rows),
        })

expected_class_rows = []
for class_id, class_name in enumerate(NAMES):
    row = {'class_id': class_id, 'class': class_name}
    for split in SPLITS:
        row[f'{split}_images'] = expected_image_counts[(split, class_name)]
        row[f'{split}_boxes'] = expected_box_counts[(split, class_name)]
    row['total_images'] = sum(row[f'{split}_images'] for split in SPLITS)
    row['total_boxes'] = sum(row[f'{split}_boxes'] for split in SPLITS)
    expected_class_rows.append(row)
expected_post_dedupe_class_df = pd.DataFrame(expected_class_rows)
post_dedupe_source_df = pd.DataFrame(post_dedupe_rows)
post_dedupe_state_sha1 = post_dedupe_state.hexdigest()
expected_post_dedupe_boxes = int(expected_post_dedupe_class_df['total_boxes'].sum())
expected_post_dedupe_class_df.to_csv(
    AUDIT_DIR / 'post_dedupe_expected_class_summary.csv', index=False
)
post_dedupe_source_df.to_csv(
    AUDIT_DIR / 'post_dedupe_source_manifest.csv', index=False
)
print('Local duplicate files removed:', len(dedupe_df))
duplicate_label_conflicts = int(dedupe_df['label_conflict'].sum()) if len(dedupe_df) else 0
print('Duplicate label conflicts resolved by split precedence:', duplicate_label_conflicts)
print('Post-dedupe source state SHA1:', post_dedupe_state_sha1)

In [ ]:
# Cell 4: Auto-orient and letterbox every local image to 640x640
# Normalized labels are assumed to describe the visually oriented image, which is
# how Roboflow and X-AnyLabeling display/export these annotations.
SOURCE_DATA_ROOT = DATA_ROOT
PREPROCESS_ROOT = Path(f'/content/galvat_trial8_preprocessed_{IMGSZ}')
PREPROCESS_MARKER = PREPROCESS_ROOT / 'PREPROCESS_COMPLETE.json'
PREPROCESS_MANIFEST = PREPROCESS_ROOT / 'preprocess_manifest.csv'
expected_preprocessed_images = sum(expected_split_images.values())

preprocess_signature = {
    'dataset_zip': str(DATASET_ZIP),
    'dataset_zip_size': DATASET_ZIP.stat().st_size,
    'target_width': IMGSZ,
    'target_height': IMGSZ,
    'method': 'exif_transpose_then_letterbox_exact_rounded_scales_v2',
    'resampling': 'LANCZOS',
    'padding_rgb': [114, 114, 114],
    'expected_images_after_dedupe': expected_preprocessed_images,
    'expected_boxes_after_dedupe': expected_post_dedupe_boxes,
    'expected_split_images': expected_split_images,
    'post_dedupe_source_state_sha1': post_dedupe_state_sha1,
    'image_quality_filtering': False,
    'normalized_label_orientation_assumption': 'labels_describe_visually_oriented_image',
}

marker_matches = False
if PREPROCESS_MARKER.exists() and PREPROCESS_MANIFEST.exists():
    prior_marker = json.loads(PREPROCESS_MARKER.read_text())
    marker_matches = all(
        prior_marker.get(key) == value
        for key, value in preprocess_signature.items()
    )
    cached_split_images = {
        split: sum(
            path.is_file() and path.suffix.lower() in IMAGE_EXTS
            for path in (PREPROCESS_ROOT / split / 'images').glob('*')
        )
        for split in SPLITS
    }
    cached_split_labels = {
        split: len(list((PREPROCESS_ROOT / split / 'labels').glob('*.txt')))
        for split in SPLITS
    }
    marker_matches = (
        marker_matches
        and cached_split_images == expected_split_images
        and cached_split_labels == expected_split_images
    )

if not marker_matches:
    if PREPROCESS_ROOT.exists():
        shutil.rmtree(PREPROCESS_ROOT)
    PREPROCESS_ROOT.mkdir(parents=True)
    preprocess_rows = []

    def save_preprocessed_image(image, output_path):
        suffix = output_path.suffix.lower()
        temporary_name = hashlib.sha1(output_path.name.encode('utf-8')).hexdigest()
        temporary = output_path.with_name(f'.{temporary_name}.part{output_path.suffix}')
        if suffix in {'.jpg', '.jpeg'}:
            image.save(temporary, format='JPEG', quality=92, subsampling=0)
        elif suffix == '.png':
            image.save(temporary, format='PNG')
        elif suffix == '.webp':
            image.save(temporary, format='WEBP', quality=92)
        elif suffix == '.bmp':
            image.save(temporary, format='BMP')
        else:
            raise ValueError(f'Unsupported image extension: {output_path}')
        temporary.replace(output_path)

    for split in SPLITS:
        output_images = PREPROCESS_ROOT / split / 'images'
        output_labels = PREPROCESS_ROOT / split / 'labels'
        output_images.mkdir(parents=True)
        output_labels.mkdir(parents=True)

        for image_path in tqdm(split_images(split), desc=f'Preprocessing {split}'):
            label_path = label_path_for_image(image_path)
            label_rows, issues = parse_label_file(label_path, split, image_path.name)
            assert not issues, issues[:10]

            with Image.open(image_path) as source_image:
                raw_width, raw_height = source_image.size
                exif_orientation = int(source_image.getexif().get(274, 1))
                oriented = ImageOps.exif_transpose(source_image).convert('RGB')

            oriented_width, oriented_height = oriented.size
            scale = min(IMGSZ / oriented_width, IMGSZ / oriented_height)
            resized_width = max(1, min(IMGSZ, round(oriented_width * scale)))
            resized_height = max(1, min(IMGSZ, round(oriented_height * scale)))
            scale_x = resized_width / oriented_width
            scale_y = resized_height / oriented_height
            pad_left = (IMGSZ - resized_width) // 2
            pad_top = (IMGSZ - resized_height) // 2

            resized = oriented.resize(
                (resized_width, resized_height),
                Image.Resampling.LANCZOS,
            )
            canvas = Image.new('RGB', (IMGSZ, IMGSZ), (114, 114, 114))
            canvas.paste(resized, (pad_left, pad_top))

            mapped_rows = []
            for class_id, x, y, width, height in label_rows:
                x1 = ((x - width / 2) * oriented_width * scale_x) + pad_left
                y1 = ((y - height / 2) * oriented_height * scale_y) + pad_top
                x2 = ((x + width / 2) * oriented_width * scale_x) + pad_left
                y2 = ((y + height / 2) * oriented_height * scale_y) + pad_top
                x1, x2 = np.clip([x1, x2], 0, IMGSZ)
                y1, y2 = np.clip([y1, y2], 0, IMGSZ)
                assert x2 > x1 and y2 > y1, (image_path, class_id, x1, y1, x2, y2)
                mapped_x = ((x1 + x2) / 2) / IMGSZ
                mapped_y = ((y1 + y2) / 2) / IMGSZ
                mapped_width = (x2 - x1) / IMGSZ
                mapped_height = (y2 - y1) / IMGSZ
                mapped_rows.append((
                    class_id,
                    mapped_x,
                    mapped_y,
                    mapped_width,
                    mapped_height,
                ))

            output_image = output_images / image_path.name
            output_label = output_labels / f'{image_path.stem}.txt'
            save_preprocessed_image(canvas, output_image)
            output_label.write_text('\n'.join(
                f'{class_id} {x:.8f} {y:.8f} {width:.8f} {height:.8f}'
                for class_id, x, y, width, height in mapped_rows
            ) + ('\n' if mapped_rows else ''))

            with Image.open(output_image) as check_image:
                assert check_image.size == (IMGSZ, IMGSZ)

            preprocess_rows.append({
                'split': split,
                'source_image': str(image_path),
                'output_image': str(output_image),
                'raw_width': raw_width,
                'raw_height': raw_height,
                'exif_orientation': exif_orientation,
                'orientation_applied': exif_orientation not in {0, 1},
                'oriented_width': oriented_width,
                'oriented_height': oriented_height,
                'resized_width': resized_width,
                'resized_height': resized_height,
                'pad_left': pad_left,
                'pad_top': pad_top,
                'scale': scale,
                'scale_x': scale_x,
                'scale_y': scale_y,
                'box_count': len(mapped_rows),
            })

    preprocess_df = pd.DataFrame(preprocess_rows)
    assert len(preprocess_df) == expected_preprocessed_images
    preprocess_df.to_csv(PREPROCESS_MANIFEST, index=False)
    PREPROCESS_MARKER.write_text(json.dumps({
        **preprocess_signature,
        'completed_at': str(datetime.datetime.now()),
        'processed_images': len(preprocess_df),
        'orientation_changes': int(preprocess_df['orientation_applied'].sum()),
    }, indent=2))
else:
    preprocess_df = pd.read_csv(PREPROCESS_MANIFEST)
    assert len(preprocess_df) == expected_preprocessed_images
    print('Using completed preprocessing cache:', PREPROCESS_ROOT)

shutil.copy2(PREPROCESS_MANIFEST, AUDIT_DIR / 'preprocess_manifest.csv')
shutil.copy2(PREPROCESS_MARKER, AUDIT_DIR / 'preprocess_summary.json')

preprocessed_cfg = dict(data_cfg)
preprocessed_cfg.update({
    'path': str(PREPROCESS_ROOT),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
})
DATA_YAML = PREPROCESS_ROOT / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump(preprocessed_cfg, sort_keys=False))
DATA_ROOT = PREPROCESS_ROOT

print('Preprocessed images:', len(preprocess_df))
print('EXIF orientation changes:', int(preprocess_df['orientation_applied'].sum()))
print('Training data.yaml:', DATA_YAML)

In [ ]:
# Cell 5: Re-audit the leakage-safe, preprocessed training copy and chart class support
def audit_dataset():
    split_rows, class_rows, bad_rows = [], [], []
    image_counts, box_counts = Counter(), Counter()
    for split in SPLITS:
        images = split_images(split)
        empty = 0
        for image_path in images:
            rows, issues = parse_label_file(label_path_for_image(image_path), split, image_path.name)
            bad_rows.extend(issues)
            if not rows:
                empty += 1
            seen = set()
            for class_id, *_ in rows:
                class_name = NAMES[class_id]
                box_counts[(split, class_name)] += 1
                seen.add(class_name)
            for class_name in seen:
                image_counts[(split, class_name)] += 1
        split_rows.append({'split': split, 'images': len(images), 'empty_label_images': empty})
    for class_id, class_name in enumerate(NAMES):
        row = {'class_id': class_id, 'class': class_name}
        for split in SPLITS:
            row[f'{split}_images'] = image_counts[(split, class_name)]
            row[f'{split}_boxes'] = box_counts[(split, class_name)]
        row['total_images'] = sum(row[f'{s}_images'] for s in SPLITS)
        row['total_boxes'] = sum(row[f'{s}_boxes'] for s in SPLITS)
        class_rows.append(row)
    return pd.DataFrame(split_rows), pd.DataFrame(class_rows), pd.DataFrame(bad_rows)

split_summary, class_summary, bad_labels = audit_dataset()
assert bad_labels.empty, f'Invalid label rows after dedupe: {len(bad_labels)}'
assert len(class_summary) == 44
assert int(split_summary['images'].sum()) == expected_preprocessed_images
assert int(split_summary['empty_label_images'].sum()) == 0
assert int(class_summary['total_boxes'].sum()) == int(preprocess_df['box_count'].sum())
comparison_columns = [
    'class_id', 'class',
    *[f'{split}_{suffix}' for split in SPLITS for suffix in ['images', 'boxes']],
    'total_images', 'total_boxes',
]
pd.testing.assert_frame_equal(
    class_summary[comparison_columns].sort_values('class_id').reset_index(drop=True),
    expected_post_dedupe_class_df[comparison_columns].sort_values('class_id').reset_index(drop=True),
    check_dtype=False,
)
assert (class_summary['train_images'] > 0).all(), class_summary[class_summary['train_images'] == 0]
assert (class_summary['valid_images'] > 0).all(), class_summary[class_summary['valid_images'] == 0]
assert (class_summary['test_images'] > 0).all(), class_summary[class_summary['test_images'] == 0]

split_summary.to_csv(AUDIT_DIR / 'training_split_summary.csv', index=False)
class_summary.to_csv(AUDIT_DIR / 'training_class_summary.csv', index=False)
shutil.copy2(DATA_YAML, AUDIT_DIR / 'training_data.yaml')
display(split_summary)
display(class_summary.sort_values('total_images'))

sns.set_theme(style='whitegrid')
plot_df = class_summary.sort_values('train_images')
plt.figure(figsize=(11, 12))
sns.barplot(data=plot_df, y='class', x='train_images', color='steelblue')
plt.title('Trial 8 training images by class after dedupe and 640px preprocessing')
plt.tight_layout()
plt.savefig(CHART_DIR / '01_training_images_by_class.png', dpi=180)
plt.show()

In [ ]:
# Cell 6: Exact GALVAT weighted metric for the Trial 8 ontology
THRESHOLDS = np.array([0.00, 0.05, 0.10, 0.20, 0.50], dtype=float)

GALVAT_RATINGS = {
    'bathroom_sign': {'fn':[8,8,8,8,9], 'fp':[7,8,8,8,9]},
    'canned_good': {'fn':[2,2,2,3,4], 'fp':[2,2,2,3,5]},
    'cellphone': {'fn':[10,10,10,10,10], 'fp':[9,9,9,9,9]},
    'chair': {'fn':[8,8,8,9,9], 'fp':[10,10,10,10,10]},
    'coffee_mug': {'fn':[8,9,9,10,10], 'fp':[5,5,5,5,6]},
    'deodorant': {'fn':[6,6,7,7,8], 'fp':[6,6,7,7,7]},
    'door': {'fn':[9,9,9,9,9], 'fp':[10,10,10,10,10]},
    'drinking_glass': {'fn':[8,8,9,9,10], 'fp':[5,5,5,5,6]},
    'electrical_outlet': {'fn':[7,7,8,8,9], 'fp':[7,7,8,8,9]},
    'exit_sign': {'fn':[7,7,7,8,8], 'fp':[7,8,8,8,9]},
    'eyeglasses': {'fn':[8,8,8,9,10], 'fp':[3,3,4,5,6]},
    'fork': {'fn':[7,7,7,8,9], 'fp':[6,7,7,8,8]},
    'hearing_aid': {'fn':[9,9,9,10,10], 'fp':[4,4,5,6,7]},
    'jug': {'fn':[6,6,6,7,8], 'fp':[5,5,6,6,7]},
    'knife': {'fn':[10,10,10,10,10], 'fp':[6,7,7,8,8]},
    'lemon': {'fn':[4,4,5,6,7], 'fp':[4,4,5,6,7]},
    'light_switch': {'fn':[7,7,8,8,9], 'fp':[5,5,5,5,6]},
    'measuring_cup': {'fn':[4,4,5,6,7], 'fp':[3,3,3,3,4]},
    'paper_towel_dispenser': {'fn':[5,5,6,7,7], 'fp':[5,5,6,7,7]},
    'pot': {'fn':[7,7,8,8,9], 'fp':[5,5,5,5,6]},
    'razor': {'fn':[8,8,8,9,9], 'fp':[6,7,7,8,8]},
    'salt_pepper_shaker': {'fn':[6,6,6,7,8], 'fp':[2,2,2,3,4]},
    'soap_dispenser': {'fn':[6,6,7,7,8], 'fp':[7,7,7,8,8]},
    'spoon': {'fn':[5,5,5,5,6], 'fp':[5,5,5,5,6]},
    'stairs': {'fn':[10,10,10,10,10], 'fp':[8,8,8,9,10]},
    'table': {'fn':[7,7,7,8,8], 'fp':[7,7,7,8,8]},
    'thermostat': {'fn':[2,2,2,2,3], 'fp':[2,2,2,2,3]},
    'toilet': {'fn':[9,9,9,10,10], 'fp':[10,10,10,10,10]},
    'toothbrush': {'fn':[6,6,7,7,8], 'fp':[7,7,8,8,9]},
    'toothpaste': {'fn':[8,8,9,9,9], 'fp':[8,8,9,9,9]},
    'trash_can': {'fn':[5,5,5,5,6], 'fp':[6,6,7,7,7]},
    'tupperware': {'fn':[3,3,3,4,5], 'fp':[2,3,3,4,5]},
    'urinal': {'fn':[9,9,9,10,10], 'fp':[10,10,10,10,10]},
    'wallet': {'fn':[9,9,9,9,9], 'fp':[10,10,10,10,10]},
    'window': {'fn':[4,4,4,5,5], 'fp':[7,8,8,8,8]},
    'wine_glass': {'fn':[8,8,9,9,10], 'fp':[5,5,5,5,6]},
    'writing_device': {'fn':[2,2,3,4,4], 'fp':[1,2,3,3,4]},
}
GALVAT_RATINGS.update({
    'keys': {'fn':[10,10,10,10,10], 'fp':[8,8,8,8,8]},
    'medicine_bottle': {'fn':[10,10,10,10,10], 'fp':[10,10,10,10,10]},
    'cane': {'fn':[10,10,10,10,10], 'fp':[8,8,8,8,8]},
    'tv_remote': {'fn':[4,4,5,6,7], 'fp':[3,3,3,4,5]},
    'purse': {'fn':[9,9,9,9,9], 'fp':[10,10,10,10,10]},
    'railing': {'fn':[8,8,8,9,9], 'fp':[7,7,8,8,9]},
    'pan': {'fn':[7,7,8,8,9], 'fp':[5,5,5,5,6]},
})

def auc(x, y):
    fn = getattr(np, 'trapezoid', np.trapz)
    return float(fn(np.asarray(y, dtype=float), np.asarray(x, dtype=float)))

def remaining_area(error_rate, ratings):
    possible = auc(THRESHOLDS, ratings)
    error = min(max(float(error_rate), 0.0), 1.0)
    if error >= 0.50:
        return 0.0, possible, 0.0
    xs = np.asarray([error, *[t for t in THRESHOLDS if t > error]])
    ys = np.interp(xs, THRESHOLDS, np.asarray(ratings, dtype=float))
    remaining = auc(xs, ys)
    return remaining, possible, remaining / possible if possible else 0.0

def add_galvat_scores(metrics_df):
    missing = sorted(set(metrics_df['class']) - set(GALVAT_RATINGS))
    assert not missing, f'Missing ratings: {missing}'
    output = []
    for row in metrics_df.to_dict('records'):
        precision = min(max(float(row['precision']), 0), 1)
        recall = min(max(float(row['recall']), 0), 1)
        fn_remaining, fn_possible, fn_score = remaining_area(1 - recall, GALVAT_RATINGS[row['class']]['fn'])
        fp_remaining, fp_possible, fp_score = remaining_area(1 - precision, GALVAT_RATINGS[row['class']]['fp'])
        row.update({
            'false_discovery_rate': 1 - precision,
            'false_negative_rate': 1 - recall,
            'fn_score': fn_score,
            'fp_score': fp_score,
            'earned_weighted_points': fn_remaining + fp_remaining,
            'possible_weight': fn_possible + fp_possible,
            'lost_weighted_points': fn_possible + fp_possible - fn_remaining - fp_remaining,
        })
        row['class_score'] = row['earned_weighted_points'] / row['possible_weight']
        output.append(row)
    result = pd.DataFrame(output)
    overall = result['earned_weighted_points'].sum() / result['possible_weight'].sum()
    return float(overall), result

assert set(NAMES) == set(GALVAT_RATINGS), sorted(set(NAMES) - set(GALVAT_RATINGS))
print('Exact GALVAT ratings loaded for all', len(NAMES), 'classes.')

In [ ]:
# Cell 7: Train YOLO26s with periodic Drive checkpoints; safely resume interruptions
TRAIN_DIR = RUN_ROOT / 'train'
LAST_PT = TRAIN_DIR / 'weights' / 'last.pt'
BEST_PT = TRAIN_DIR / 'weights' / 'best.pt'
COMPLETE_MARKER = TRAIN_DIR / 'TRAINING_COMPLETE.json'

train_args = dict(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=32,
    device=0,
    workers=8,
    cache='disk',
    optimizer='AdamW',
    lr0=0.0008,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5.0,
    patience=30,
    seed=SEED,
    deterministic=True,
    pretrained=True,
    freeze=0,
    cos_lr=True,
    mosaic=0.25,
    close_mosaic=15,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    erasing=0.0,
    auto_augment=None,
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.25,
    degrees=0.0,
    translate=0.05,
    scale=0.25,
    shear=0.0,
    perspective=0.0,
    save=True,
    save_period=SAVE_PERIOD,
    plots=True,
    amp=True,
    project=str(RUN_ROOT),
    name='train',
    exist_ok=True,
    verbose=True,
)
train_configuration = {
    'model_init': MODEL_INIT,
    'preprocess_signature': preprocess_signature,
    'train_args': {key: str(value) for key, value in train_args.items()},
}
TRAIN_CONFIG_SHA1 = hashlib.sha1(
    json.dumps(train_configuration, sort_keys=True).encode('utf-8')
).hexdigest()
train_configuration['configuration_sha1'] = TRAIN_CONFIG_SHA1
TRAIN_CONFIG_PATH = RUN_ROOT / 'train_configuration.json'
if TRAIN_CONFIG_PATH.exists():
    prior_train_configuration = json.loads(TRAIN_CONFIG_PATH.read_text())
    assert prior_train_configuration.get('configuration_sha1') == TRAIN_CONFIG_SHA1, (
        'The stable Drive run folder contains a different training configuration. '
        'Increment RUN_NAME before starting a different experiment.'
    )
else:
    TRAIN_CONFIG_PATH.write_text(json.dumps(train_configuration, indent=2))

if COMPLETE_MARKER.exists():
    completed_training = json.loads(COMPLETE_MARKER.read_text())
    assert completed_training.get('configuration_sha1') == TRAIN_CONFIG_SHA1
    print('Training already completed:', COMPLETE_MARKER)
elif LAST_PT.exists():
    print('Resuming interrupted run from:', LAST_PT)
    model = YOLO(str(LAST_PT))
    model.train(resume=True)
    COMPLETE_MARKER.write_text(json.dumps({
        'resumed': True,
        'completed_at': str(datetime.datetime.now()),
        'configuration_sha1': TRAIN_CONFIG_SHA1,
    }, indent=2))
else:
    print('Starting:', MODEL_INIT)
    model = YOLO(MODEL_INIT)
    model.train(**train_args)
    COMPLETE_MARKER.write_text(json.dumps({
        'resumed': False,
        'completed_at': str(datetime.datetime.now()),
        'configuration_sha1': TRAIN_CONFIG_SHA1,
    }, indent=2))

assert BEST_PT.exists(), f'Missing: {BEST_PT}'
assert LAST_PT.exists(), f'Missing: {LAST_PT}'
print('BEST_PT:', BEST_PT)
print('LAST_PT:', LAST_PT)

In [ ]:
# Cell 8: Exact fixed-threshold detector evaluation helpers
def load_ground_truth(split):
    cache = {}
    for image_path in split_images(split):
        rows, issues = parse_label_file(label_path_for_image(image_path), split, image_path.name)
        assert not issues
        by_class = defaultdict(list)
        for row in rows:
            by_class[row[0]].append(xywh_to_xyxy(row))
        cache[image_path.name] = {
            class_id: np.asarray(boxes, dtype=float)
            for class_id, boxes in by_class.items()
        }
    return cache

GT_CACHE = {split: load_ground_truth(split) for split in ['valid', 'test']}

def collect_predictions(model_path, split, nms_iou, source_paths=None):
    paths = list(source_paths) if source_paths is not None else split_images(split)
    assert paths, f'No prediction images for split: {split}'
    path_keys = [Path(path).name for path in paths]
    assert len(set(path_keys)) == len(path_keys), (
        f'Duplicate image basenames in {split}; filename-keyed evaluation is ambiguous'
    )
    predictions = {}
    detector = None
    progress = tqdm(total=len(paths), desc=f'Predict {split} iou={nms_iou}')
    try:
        detector = YOLO(str(model_path))
        for start in range(0, len(paths), PRED_BATCH):
            chunk = paths[start:start + PRED_BATCH]
            chunk_results = detector.predict(
                source=[str(path) for path in chunk],
                stream=False,
                imgsz=IMGSZ,
                conf=PRED_MIN_CONF,
                iou=float(nms_iou),
                max_det=MAX_DET,
                device=0,
                batch=len(chunk),
                rect=False,
                agnostic_nms=False,
                verbose=False,
            )
            assert len(chunk_results) == len(chunk)
            # Ultralytics may retain stale result.path metadata when the same
            # predictor receives repeated path-list calls. Results preserve
            # input order, so key each result from its paired source path.
            for source_path, result in zip(chunk, chunk_results):
                key = Path(source_path).name
                assert key not in predictions, f'Duplicate prediction key: {key}'
                if result.boxes is None or len(result.boxes) == 0:
                    predictions[key] = {
                        'class_id': np.empty(0, dtype=int),
                        'confidence': np.empty(0, dtype=float),
                        'boxes': np.empty((0, 4), dtype=float),
                    }
                else:
                    predictions[key] = {
                        'class_id': result.boxes.cls.detach().cpu().numpy().astype(int),
                        'confidence': result.boxes.conf.detach().cpu().numpy().astype(float),
                        'boxes': result.boxes.xyxyn.detach().cpu().numpy().astype(float),
                    }
            del chunk_results
            progress.update(len(chunk))
    finally:
        progress.close()
        if detector is not None and getattr(detector, 'model', None) is not None:
            detector.model.to('cpu')
        del detector
        gc.collect()
        torch.cuda.empty_cache()
    assert len(predictions) == len(paths), (
        f'Prediction key collision or missing result: {len(predictions)} / {len(paths)}'
    )
    return predictions

def pairwise_iou(pred_box, gt_boxes):
    if len(gt_boxes) == 0:
        return np.empty(0, dtype=float)
    top_left = np.maximum(pred_box[:2], gt_boxes[:, :2])
    bottom_right = np.minimum(pred_box[2:], gt_boxes[:, 2:])
    wh = np.maximum(0.0, bottom_right - top_left)
    inter = wh[:, 0] * wh[:, 1]
    pred_area = max(0.0, pred_box[2] - pred_box[0]) * max(0.0, pred_box[3] - pred_box[1])
    gt_area = np.maximum(0.0, gt_boxes[:, 2] - gt_boxes[:, 0]) * np.maximum(0.0, gt_boxes[:, 3] - gt_boxes[:, 1])
    union = pred_area + gt_area - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)

def match_one_class(gt_boxes, pred_boxes, pred_conf, threshold):
    keep = pred_conf >= float(threshold)
    pred_boxes = pred_boxes[keep]
    pred_conf = pred_conf[keep]
    order = np.argsort(-pred_conf)
    matched = np.zeros(len(gt_boxes), dtype=bool)
    tp = fp = 0
    for pred_box in pred_boxes[order]:
        ious = pairwise_iou(pred_box, gt_boxes)
        if len(ious):
            ious[matched] = -1
            best = int(np.argmax(ious))
            if ious[best] >= MATCH_IOU:
                matched[best] = True
                tp += 1
                continue
        fp += 1
    return tp, fp, int((~matched).sum())

def evaluate_thresholds(gt_cache, prediction_cache, confidence_values):
    rows = []
    for threshold in confidence_values:
        counts = {class_id: [0, 0, 0] for class_id in range(len(NAMES))}
        for image_name, gt_by_class in gt_cache.items():
            pred = prediction_cache.get(image_name, {
                'class_id': np.empty(0, dtype=int),
                'confidence': np.empty(0),
                'boxes': np.empty((0, 4)),
            })
            active = set(gt_by_class) | set(pred['class_id'].tolist())
            for class_id in active:
                gt_boxes = gt_by_class.get(class_id, np.empty((0, 4)))
                mask = pred['class_id'] == class_id
                tp, fp, fn = match_one_class(
                    gt_boxes, pred['boxes'][mask], pred['confidence'][mask], threshold
                )
                counts[class_id][0] += tp
                counts[class_id][1] += fp
                counts[class_id][2] += fn
        metric_rows = []
        for class_id, class_name in enumerate(NAMES):
            tp, fp, fn = counts[class_id]
            metric_rows.append({
                'class_id': class_id,
                'class': class_name,
                'confidence_threshold': float(threshold),
                'tp': tp, 'fp': fp, 'fn': fn,
                'precision': tp / (tp + fp) if tp + fp else 0.0,
                'recall': tp / (tp + fn) if tp + fn else 0.0,
            })
        score, scored = add_galvat_scores(pd.DataFrame(metric_rows))
        scored['overall_global_galvat'] = score
        rows.append(scored)
    return pd.concat(rows, ignore_index=True)

def select_per_class_thresholds(all_threshold_rows):
    selected = []
    for class_name, group in all_threshold_rows.groupby('class'):
        best = group.sort_values(
            ['class_score', 'recall', 'precision', 'confidence_threshold'],
            ascending=[False, False, False, False],
        ).iloc[0]
        selected.append(best)
    selected_df = pd.DataFrame(selected).sort_values('class_id')
    overall = selected_df['earned_weighted_points'].sum() / selected_df['possible_weight'].sum()
    return float(overall), selected_df

def evaluate_threshold_map(gt_cache, prediction_cache, thresholds_by_class):
    thresholds = sorted(set(float(v) for v in thresholds_by_class.values()))
    all_rows = evaluate_thresholds(gt_cache, prediction_cache, thresholds)
    selected = []
    for class_name in NAMES:
        threshold = float(thresholds_by_class[class_name])
        row = all_rows[(all_rows['class'] == class_name) & np.isclose(all_rows['confidence_threshold'], threshold)].iloc[0]
        selected.append(row)
    result = pd.DataFrame(selected).sort_values('class_id')
    overall = result['earned_weighted_points'].sum() / result['possible_weight'].sum()
    return float(overall), result

In [ ]:
# Cell 9: Restart-safe validation checkpoint, NMS, and class-threshold selection
# A completed training call can leave its trainer/model attached to the notebook.
if 'model' in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
print(
    'CUDA before selection (allocated/reserved GB):',
    round(torch.cuda.memory_allocated() / 1e9, 3),
    round(torch.cuda.memory_reserved() / 1e9, 3),
)

def epoch_number(path):
    digits = ''.join(ch for ch in path.stem if ch.isdigit())
    return int(digits) if digits else 10**9

periodic = sorted((TRAIN_DIR / 'weights').glob('epoch*.pt'), key=epoch_number)
checkpoint_candidates = []
checkpoint_sha1_by_path = {}
seen_checkpoint_hashes = set()
for path in [*periodic, BEST_PT, LAST_PT]:
    if not path.exists():
        continue
    checkpoint_sha1 = sha1_file(path)
    if checkpoint_sha1 in seen_checkpoint_hashes:
        print('Skipping byte-identical checkpoint:', path.name)
        continue
    seen_checkpoint_hashes.add(checkpoint_sha1)
    checkpoint_candidates.append(path)
    checkpoint_sha1_by_path[str(path)] = checkpoint_sha1
assert checkpoint_candidates
print('Checkpoint candidates:', [p.name for p in checkpoint_candidates])

global_rows = []
class_candidate_rows = []
leaderboard_rows = []
PAIR_CACHE_DIR = SELECTION_DIR / 'pair_cache'
PAIR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for checkpoint in checkpoint_candidates:
    checkpoint_sha1 = checkpoint_sha1_by_path[str(checkpoint)]
    for nms_iou in NMS_IOU_VALUES:
        pair_name = (
            f'{checkpoint.stem}_{checkpoint_sha1[:12]}_'
            f'nms_{str(nms_iou).replace(".", "p")}'
        )
        pair_dir = PAIR_CACHE_DIR / pair_name
        pair_dir.mkdir(parents=True, exist_ok=True)
        pair_csv = pair_dir / 'validation_per_class_thresholds.csv'
        pair_done = pair_dir / 'COMPLETE.json'

        if pair_csv.exists() and pair_done.exists():
            threshold_rows = pd.read_csv(pair_csv)
            print('Loaded cached validation pair:', pair_name)
        else:
            predictions = collect_predictions(checkpoint, 'valid', nms_iou)
            threshold_rows = evaluate_thresholds(GT_CACHE['valid'], predictions, CONF_VALUES)
            threshold_rows['checkpoint'] = str(checkpoint)
            threshold_rows['checkpoint_name'] = checkpoint.name
            threshold_rows['checkpoint_sha1'] = checkpoint_sha1
            threshold_rows['nms_iou'] = nms_iou
            temporary_csv = pair_csv.with_suffix('.csv.part')
            threshold_rows.to_csv(temporary_csv, index=False)
            temporary_csv.replace(pair_csv)
            pair_done.write_text(json.dumps({
                'checkpoint': str(checkpoint),
                'checkpoint_sha1': checkpoint_sha1,
                'nms_iou': nms_iou,
                'completed_at': str(datetime.datetime.now()),
            }, indent=2))
            del predictions
            torch.cuda.empty_cache()

        assert len(threshold_rows) == len(NAMES) * len(CONF_VALUES)
        class_candidate_rows.append(threshold_rows)

        global_summary = (
            threshold_rows.groupby('confidence_threshold', as_index=False)
            .agg(
                galvat=('overall_global_galvat', 'first'),
                macro_precision=('precision', 'mean'),
                macro_recall=('recall', 'mean'),
                total_tp=('tp', 'sum'), total_fp=('fp', 'sum'), total_fn=('fn', 'sum'),
            )
        )
        global_summary['checkpoint'] = str(checkpoint)
        global_summary['checkpoint_name'] = checkpoint.name
        global_summary['nms_iou'] = nms_iou
        global_rows.append(global_summary)

        classwise_score, classwise = select_per_class_thresholds(threshold_rows)
        best_global = global_summary.sort_values(
            ['galvat', 'macro_recall', 'macro_precision'], ascending=False
        ).iloc[0]
        leaderboard_rows.append({
            'checkpoint': str(checkpoint),
            'checkpoint_name': checkpoint.name,
            'nms_iou': nms_iou,
            'per_class_threshold_galvat': classwise_score,
            'per_class_macro_precision': float(classwise['precision'].mean()),
            'per_class_macro_recall': float(classwise['recall'].mean()),
            'best_global_threshold': float(best_global['confidence_threshold']),
            'best_global_galvat': float(best_global['galvat']),
        })
        del threshold_rows
        gc.collect()

validation_global_df = pd.concat(global_rows, ignore_index=True)
validation_classes_df = pd.concat(class_candidate_rows, ignore_index=True)
leaderboard_df = pd.DataFrame(leaderboard_rows).sort_values(
    ['per_class_threshold_galvat', 'per_class_macro_recall', 'best_global_galvat'],
    ascending=False,
)
validation_global_df.to_csv(SELECTION_DIR / 'validation_global_threshold_sweep.csv', index=False)
validation_classes_df.to_csv(SELECTION_DIR / 'validation_per_class_threshold_candidates.csv', index=False)
leaderboard_df.to_csv(SELECTION_DIR / 'validation_checkpoint_nms_leaderboard.csv', index=False)
display(leaderboard_df)

winner = leaderboard_df.iloc[0]
SELECTED_PT = Path(winner['checkpoint'])
SELECTED_NMS_IOU = float(winner['nms_iou'])
SELECTED_GLOBAL_CONF = float(winner['best_global_threshold'])
selected_candidates = validation_classes_df[
    (validation_classes_df['checkpoint'] == str(SELECTED_PT))
    & np.isclose(validation_classes_df['nms_iou'], SELECTED_NMS_IOU)
]
SELECTED_VAL_GALVAT, selected_threshold_rows = select_per_class_thresholds(selected_candidates)
CLASS_THRESHOLDS = {
    row['class']: float(row['confidence_threshold'])
    for row in selected_threshold_rows.to_dict('records')
}
SELECTED_PT_SHA1 = sha1_file(SELECTED_PT)
THRESHOLD_POLICY_SHA1 = hashlib.sha1(
    json.dumps(CLASS_THRESHOLDS, sort_keys=True).encode('utf-8')
).hexdigest()
selected_threshold_rows.to_csv(SELECTION_DIR / 'selected_per_class_thresholds_and_val_metrics.csv', index=False)
(SELECTION_DIR / 'selected_per_class_thresholds.json').write_text(json.dumps(CLASS_THRESHOLDS, indent=2))

selection = {
    'checkpoint': str(SELECTED_PT),
    'checkpoint_name': SELECTED_PT.name,
    'checkpoint_sha1': SELECTED_PT_SHA1,
    'nms_iou': SELECTED_NMS_IOU,
    'best_global_conf': SELECTED_GLOBAL_CONF,
    'best_global_val_galvat': float(winner['best_global_galvat']),
    'per_class_threshold_val_galvat': SELECTED_VAL_GALVAT,
    'match_iou': MATCH_IOU,
    'threshold_policy_sha1': THRESHOLD_POLICY_SHA1,
    'selected_on': 'validation_only',
}
(SELECTION_DIR / 'selection.json').write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))

In [ ]:
# Cell 10: Final validation report with exact selected thresholds and official AP
def official_ap_dataframe(metrics):
    summary = pd.DataFrame(metrics.summary())
    if summary.empty:
        return pd.DataFrame({'class': NAMES, 'ap50': 0.0, 'map50_95': 0.0})
    renamed = summary.rename(columns={
        'Class': 'class', 'mAP50': 'ap50', 'mAP50-95': 'map50_95',
        'Images': 'official_images', 'Instances': 'official_instances',
    })
    return renamed[['class', 'ap50', 'map50_95', 'official_images', 'official_instances']]

selected_model = YOLO(str(SELECTED_PT))
official_val = selected_model.val(
    data=str(DATA_YAML), split='val', imgsz=IMGSZ, conf=0.001,
    iou=SELECTED_NMS_IOU, device=0, plots=True, save_json=False,
    project=str(SELECTION_DIR), name='official_val_ap', exist_ok=True, verbose=False,
)
val_predictions = collect_predictions(SELECTED_PT, 'valid', SELECTED_NMS_IOU)
val_galvat, val_exact = evaluate_threshold_map(GT_CACHE['valid'], val_predictions, CLASS_THRESHOLDS)
val_report = val_exact.merge(official_ap_dataframe(official_val), on='class', how='left')
val_report = val_report.merge(
    class_summary[['class', 'valid_images', 'valid_boxes']], on='class', how='left'
).sort_values('class_score')
val_report[['ap50', 'map50_95']] = val_report[['ap50', 'map50_95']].fillna(0.0)
val_report.to_csv(SELECTION_DIR / 'validation_selected_per_class_metrics.csv', index=False)
assert len(val_report) == 44
assert abs(val_galvat - SELECTED_VAL_GALVAT) < 1e-12
print('Frozen validation GALVAT:', val_galvat)
display(val_report)

In [ ]:
# Cell 11: One-shot test evaluation using frozen validation settings
TEST_LOCK = TEST_DIR / 'TEST_EVALUATION_COMPLETE.json'
frozen_test_signature = {
    'checkpoint_sha1': SELECTED_PT_SHA1,
    'threshold_policy_sha1': THRESHOLD_POLICY_SHA1,
    'nms_iou': SELECTED_NMS_IOU,
    'match_iou': MATCH_IOU,
    'preprocess_source_state_sha1': post_dedupe_state_sha1,
}
if TEST_LOCK.exists():
    prior_test_lock = json.loads(TEST_LOCK.read_text())
    assert all(
        prior_test_lock.get(key) == value
        for key, value in frozen_test_signature.items()
    ), (
        'The frozen test lock belongs to different model/data/threshold settings. '
        'Increment RUN_NAME; do not overwrite or retune an existing test evaluation.'
    )
    test_report = pd.read_csv(TEST_DIR / 'test_frozen_per_class_metrics.csv')
    test_summary = json.loads((TEST_DIR / 'test_frozen_summary.json').read_text())
    TEST_GALVAT = float(test_summary['galvat_weighted_score'])
    print('Loaded completed frozen test evaluation.')
else:
    test_predictions = collect_predictions(SELECTED_PT, 'test', SELECTED_NMS_IOU)
    TEST_GALVAT, test_exact = evaluate_threshold_map(GT_CACHE['test'], test_predictions, CLASS_THRESHOLDS)
    official_test = selected_model.val(
        data=str(DATA_YAML), split='test', imgsz=IMGSZ, conf=0.001,
        iou=SELECTED_NMS_IOU, device=0, plots=True, save_json=True,
        project=str(TEST_DIR), name='official_test_ap', exist_ok=True, verbose=False,
    )
    test_report = test_exact.merge(official_ap_dataframe(official_test), on='class', how='left')
    test_report = test_report.merge(
        class_summary[['class', 'test_images', 'test_boxes']], on='class', how='left'
    ).sort_values('class_score')
    test_report[['ap50', 'map50_95']] = test_report[['ap50', 'map50_95']].fillna(0.0)
    test_report.to_csv(TEST_DIR / 'test_frozen_per_class_metrics.csv', index=False)
    test_summary = {
        'galvat_weighted_score': TEST_GALVAT,
        'macro_precision': float(test_report['precision'].mean()),
        'macro_recall': float(test_report['recall'].mean()),
        'macro_ap50': float(test_report['ap50'].mean()),
        'macro_map50_95': float(test_report['map50_95'].mean()),
        'classes': len(test_report),
        'classes_below_0_2': int((test_report['class_score'] < 0.2).sum()),
        'zero_score_classes': int((test_report['class_score'] <= 0).sum()),
        'checkpoint': str(SELECTED_PT),
        'nms_iou': SELECTED_NMS_IOU,
        'threshold_policy': 'validation_selected_per_class',
        'test_tuning_performed': False,
    }
    (TEST_DIR / 'test_frozen_summary.json').write_text(json.dumps(test_summary, indent=2))
    TEST_LOCK.write_text(json.dumps({
        **frozen_test_signature,
        'completed_at': str(datetime.datetime.now()),
    }, indent=2))

print(json.dumps(test_summary, indent=2))
print('Weakest classes:')
display(test_report.head(15))

In [ ]:
# Cell 12: Frozen-threshold spoken burden and fresh background-RoI evaluation
def accepted_prediction_counts(prediction_cache, thresholds):
    rows = []
    for image_name, pred in prediction_cache.items():
        accepted = []
        for class_id, confidence in zip(pred['class_id'], pred['confidence']):
            class_name = NAMES[int(class_id)]
            if confidence >= thresholds[class_name]:
                accepted.append((class_name, float(confidence)))
        rows.append({
            'image': image_name,
            'accepted_detections': len(accepted),
            'classes': ','.join(x[0] for x in accepted),
        })
    return pd.DataFrame(rows)

if 'test_predictions' not in globals():
    test_predictions = collect_predictions(SELECTED_PT, 'test', SELECTED_NMS_IOU)
spoken_df = accepted_prediction_counts(test_predictions, CLASS_THRESHOLDS)
spoken_df.to_csv(TEST_DIR / 'test_spoken_detection_counts.csv', index=False)
SPOKEN_BUDGET = 3

BG_DIR = RUN_ROOT / 'fresh_background_roi'
BG_IMAGES = BG_DIR / 'images'
BG_IMAGES.mkdir(parents=True, exist_ok=True)

def create_background_crops(target=400, crop_size=320, seed=91):
    rng = random.Random(seed)
    candidates = split_images('test')
    rng.shuffle(candidates)
    rows = []
    for image_path in tqdm(candidates, desc='Creating non-overlap background crops'):
        if len(rows) >= target:
            break
        gt_rows, _ = parse_label_file(label_path_for_image(image_path))
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception:
            continue
        width, height = image.size
        size = min(crop_size, width, height)
        if size < 128:
            continue
        gt_abs = []
        for row in gt_rows:
            box = xywh_to_xyxy(row)
            gt_abs.append(np.array([box[0]*width, box[1]*height, box[2]*width, box[3]*height]))
        for _ in range(30):
            x1 = rng.randint(0, width - size)
            y1 = rng.randint(0, height - size)
            crop_box = np.array([x1, y1, x1 + size, y1 + size])
            def intersects(a, b):
                return min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1])
            if any(intersects(crop_box, gt) for gt in gt_abs):
                continue
            out_path = BG_IMAGES / f'fresh_bg_{len(rows):04d}_{image_path.stem[:60]}.jpg'
            image.crop(tuple(crop_box)).resize((IMGSZ, IMGSZ), Image.Resampling.LANCZOS).save(out_path, quality=92)
            rows.append({'crop': str(out_path), 'source': str(image_path), 'x1': x1, 'y1': y1, 'size': size})
            break
    return pd.DataFrame(rows)

bg_manifest = create_background_crops()
assert len(bg_manifest) >= 100, f'Only created {len(bg_manifest)} background crops.'
bg_manifest.to_csv(BG_DIR / 'background_crop_manifest.csv', index=False)
bg_paths = [Path(path) for path in bg_manifest['crop']]
bg_predictions = collect_predictions(SELECTED_PT, 'fresh_background', SELECTED_NMS_IOU, bg_paths)
bg_counts = accepted_prediction_counts(bg_predictions, CLASS_THRESHOLDS)
bg_counts.to_csv(BG_DIR / 'background_predictions.csv', index=False)

roi_summary = {
    'fresh_background_crop_count': len(bg_counts),
    'fresh_background_any_detection_rate': float((bg_counts['accepted_detections'] > 0).mean()),
    'fresh_background_mean_detections': float(bg_counts['accepted_detections'].mean()),
    'test_spoken_budget': SPOKEN_BUDGET,
    'test_over_budget_rate': float((spoken_df['accepted_detections'] > SPOKEN_BUDGET).mean()),
    'test_mean_accepted_detections': float(spoken_df['accepted_detections'].mean()),
    'settings_frozen_from_validation': True,
}
(RUN_ROOT / 'roi_spoken_summary.json').write_text(json.dumps(roi_summary, indent=2))
print(json.dumps(roi_summary, indent=2))

In [ ]:
# Cell 13: Decision-oriented charts and complete per-class tables
plt.figure(figsize=(10, 6))
sns.barplot(data=leaderboard_df.head(15), y='checkpoint_name', x='per_class_threshold_galvat', hue='nms_iou', dodge=False)
plt.title('Validation GALVAT checkpoint and NMS ranking')
plt.tight_layout()
plt.savefig(CHART_DIR / '02_validation_checkpoint_ranking.png', dpi=180)
plt.show()

plot_test = test_report.sort_values('class_score')
plt.figure(figsize=(11, 12))
sns.barplot(data=plot_test, y='class', x='class_score', color='seagreen')
plt.axvline(0.5, color='firebrick', linestyle='--')
plt.title(f'Trial 8 frozen test GALVAT by class (overall={TEST_GALVAT:.3f})')
plt.tight_layout()
plt.savefig(CHART_DIR / '03_test_galvat_by_class.png', dpi=180)
plt.show()

plt.figure(figsize=(11, 12))
sns.barplot(data=test_report.sort_values('lost_weighted_points', ascending=False), y='class', x='lost_weighted_points', color='firebrick')
plt.title('Test lost GALVAT weighted points by class')
plt.tight_layout()
plt.savefig(CHART_DIR / '04_test_lost_weighted_points.png', dpi=180)
plt.show()

threshold_plot = selected_threshold_rows.sort_values('confidence_threshold')
plt.figure(figsize=(11, 12))
sns.barplot(data=threshold_plot, y='class', x='confidence_threshold', color='darkorange')
plt.xlim(0, 1)
plt.title('Validation-selected confidence threshold by class')
plt.tight_layout()
plt.savefig(CHART_DIR / '05_selected_class_thresholds.png', dpi=180)
plt.show()

plt.figure(figsize=(8, 7))
sns.scatterplot(data=test_report, x='recall', y='precision', size='possible_weight', hue='class_score', palette='viridis', sizes=(40, 250))
plt.title('Frozen test precision-recall by class')
plt.tight_layout()
plt.savefig(CHART_DIR / '06_test_precision_recall.png', dpi=180)
plt.show()

comparison_summary = pd.DataFrame([
    {'run': 'Trial 7 corrected full ontology', 'galvat': 0.3786278257},
    {'run': 'Trial 7 corrected supported classes', 'galvat': 0.5040794011},
    {'run': 'Trial 8 validation class thresholds', 'galvat': SELECTED_VAL_GALVAT},
    {'run': 'Trial 8 frozen test', 'galvat': TEST_GALVAT},
])
comparison_summary.to_csv(RUN_ROOT / 'trial7_trial8_galvat_comparison.csv', index=False)
display(comparison_summary)

In [ ]:
# Cell 14: Copy the selected checkpoint, export ONNX, and smoke-test inference
SELECTED_COPY = EXPORT_DIR / 'galvat_trial8_selected.pt'
shutil.copy2(SELECTED_PT, SELECTED_COPY)
shutil.copy2(SELECTION_DIR / 'selected_per_class_thresholds.json', EXPORT_DIR / 'selected_per_class_thresholds.json')

export_model = YOLO(str(SELECTED_COPY))
exported = Path(export_model.export(
    format='onnx', imgsz=IMGSZ, opset=12, simplify=True,
    dynamic=False, nms=False, batch=1, device=0,
))
ONNX_PATH = EXPORT_DIR / 'galvat_trial8_selected.onnx'
if exported.resolve() != ONNX_PATH.resolve():
    shutil.copy2(exported, ONNX_PATH)
assert ONNX_PATH.exists() and ONNX_PATH.stat().st_size > 0

smoke_image = split_images('test')[0]
onnx_model = YOLO(str(ONNX_PATH), task='detect')
smoke_results = onnx_model.predict(source=str(smoke_image), imgsz=IMGSZ, conf=PRED_MIN_CONF, verbose=False)
assert len(smoke_results) == 1
export_summary = {
    'pytorch': str(SELECTED_COPY),
    'pytorch_sha1': sha1_file(SELECTED_COPY),
    'onnx': str(ONNX_PATH),
    'onnx_sha1': sha1_file(ONNX_PATH),
    'onnx_size_mb': ONNX_PATH.stat().st_size / 1e6,
    'imgsz': IMGSZ,
    'opset': 12,
    'smoke_test_image': str(smoke_image),
    'smoke_test_passed': True,
}
(EXPORT_DIR / 'export_summary.json').write_text(json.dumps(export_summary, indent=2))
print(json.dumps(export_summary, indent=2))

In [ ]:
# Cell 15: Final reproducible run summary (all artifacts already live on Drive)
final_summary = {
    'run_name': RUN_NAME,
    'run_id': 'galvat_trial8_yolo26s_640_galvat_selected_planned_v1',
    'created_at': RUN_STAMP,
    'dataset_manifest_id': 'galvat_trial8_worker_manual_merged_20260803_v1',
    'experiment_plan_id': 'galvat_trial8_galvat_optimized_train_eval_export_v1',
    'dataset_zip': str(DATASET_ZIP),
    'source_checkpoint_summary': checkpoint_summary,
    'local_training_images_after_exact_dedupe': int(split_summary['images'].sum()),
    'local_exact_duplicates_removed': int(len(dedupe_df)),
    'duplicate_label_conflicts_resolved_by_split_precedence': duplicate_label_conflicts,
    'cross_split_duplicate_hash_groups_detected': cross_split_hashes,
    'post_dedupe_source_state_sha1': post_dedupe_state_sha1,
    'preprocessing': preprocess_signature,
    'preprocessed_images': int(len(preprocess_df)),
    'images_auto_oriented': int(preprocess_df['orientation_applied'].sum()),
    'model_init': MODEL_INIT,
    'imgsz': IMGSZ,
    'training_configuration_sha1': TRAIN_CONFIG_SHA1,
    'selected_checkpoint': str(SELECTED_PT),
    'selected_checkpoint_sha1': SELECTED_PT_SHA1,
    'selected_model_copy': str(SELECTED_COPY),
    'selected_onnx': str(ONNX_PATH),
    'selected_nms_iou': SELECTED_NMS_IOU,
    'selected_global_conf_comparator': SELECTED_GLOBAL_CONF,
    'threshold_policy': 'validation_selected_per_class',
    'threshold_policy_sha1': THRESHOLD_POLICY_SHA1,
    'validation_galvat': SELECTED_VAL_GALVAT,
    'test_galvat': TEST_GALVAT,
    'test_macro_precision': test_summary['macro_precision'],
    'test_macro_recall': test_summary['macro_recall'],
    'test_macro_ap50': test_summary['macro_ap50'],
    'test_macro_map50_95': test_summary['macro_map50_95'],
    'fresh_background_any_detection_rate': roi_summary['fresh_background_any_detection_rate'],
    'test_spoken_over_budget_rate': roi_summary['test_over_budget_rate'],
    'test_tuning_performed': False,
    'primary_metric': 'full-ontology GALVAT weighted score',
    'trial7_corrected_full_ontology_baseline': 0.3786278257,
    'trial7_corrected_supported_baseline': 0.5040794011,
    'remaining_validity_risk': (
        'Exact-image leakage is removed locally. Near-duplicate and source/scene leakage '
        'remain possible because the checkpoint lacks complete source_id and scene_id metadata. '
        'Normalized annotations are assumed to describe each visually oriented source image.'
    ),
    'deployment_target': 'Magic Leap; YOLO26s selected because YOLO26m exceeds the size budget',
}
(RUN_ROOT / 'trial8_final_run_summary.json').write_text(json.dumps(final_summary, indent=2))

# Small artifact archive for convenient sharing; weights remain separately in export/.
METRICS_ZIP = RUN_ROOT / 'trial8_metrics_and_reports.zip'
with zipfile.ZipFile(METRICS_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_ROOT.rglob('*'):
        if path.is_file() and path != METRICS_ZIP and path.suffix.lower() in {'.csv', '.json', '.png', '.yaml'}:
            archive.write(path, path.relative_to(RUN_ROOT))
with zipfile.ZipFile(METRICS_ZIP) as archive:
    assert archive.testzip() is None

print(json.dumps(final_summary, indent=2))
print('\nFinal Drive run folder:')
print(RUN_ROOT)
print('\nCells are restart-safe. Rerun all cells to resume the same Drive run.')
print('Do not change thresholds after viewing test results.')